# NC-Musical: AI Music Transcription (Batch Colab GPU Pipeline)
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine directly inside Google Colab.

**Hardware Acceleration:** NVIDIA GPUs (T4, L4, A100)

**Key Features:**
- **Batch Transcription:** Process single or multiple YouTube URLs / uploaded files sequentially.
- **Original Song Naming:** Automatically names MIDI and chord files after the original YouTube song title or uploaded filename.
- **Dynamic Prompts:** No hardcoded links — easily paste links or be prompted on run.
- **Future-Proof yt-dlp:** Direct live master-branch builds for dependable extraction.
- **SoundFont Previews:** In-notebook audio playback via `MS Basic.sf3`.
- **Automated Chord Analysis:** Extracts and timestamps harmonic progressions using `music21`.
- **Automatic Downloads & ZIP:** Triggers immediate browser downloads and bundles batch results into `.zip`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [1]:
# @title 1. (Optional) Connect Google Drive for Persistent Model Storage
# @markdown Connecting Google Drive saves downloaded AI models and SoundFonts permanently so you don't re-download them across Colab restarts.
# @markdown Set **use_google_drive** to `False` to skip the authorization prompt and use fast session temporary storage instead.
use_google_drive = True # @param {type:"boolean"}
import os

if use_google_drive:
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive is already connected and active!")
    else:
        try:
            from google.colab import drive
            print("Connecting to Google Drive...")
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"Notice: Google Drive mount skipped ({e}). Falling back to local session storage.")

if os.path.exists('/content/drive/MyDrive'):
    drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
    os.makedirs(drive_models_dir, exist_ok=True)
    hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")
    print(f"✓ Persistent Model Storage enabled: {drive_models_dir}")
else:
    local_cache_dir = "/content/cache/huggingface"
    os.makedirs(local_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = local_cache_dir
    print("ℹ Session Storage active: Models will be cached in /content/cache/ for this session.")


Connecting to Google Drive...
Mounted at /content/drive
✓ Persistent Model Storage enabled: /content/drive/MyDrive/NC-Musical-Models


In [2]:
# @title 2. Check Hardware (GPU / TPU v5e-1), Tokens & Setup Repository
import os
import sys
import subprocess
import getpass
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Checking hardware accelerator environment...")
has_gpu = torch.cuda.is_available()
has_tpu = False

if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ NVIDIA GPU Detected: {gpu_name}")
    !nvidia-smi
else:
    # Check for Google Colab TPU v5e / TPU v2/v3 environment
    if 'COLAB_TPU_ADDR' in os.environ or os.environ.get('PJRT_DEVICE') == 'TPU' or os.path.exists('/dev/accel0'):
        has_tpu = True
        os.environ['PJRT_DEVICE'] = 'TPU'
        print('✓ Google Cloud TPU (v5e-1 / PJRT) Detected!')
    else:
        try:
            import torch_xla.core.xla_model as xm
            has_tpu = True
            print(f'✓ Google Cloud TPU Detected: {xm.xla_device()}')
        except Exception:
            print('ℹ Standard CPU environment detected (No GPU/TPU found).')

# Configure HuggingFace Token (HF_TOKEN)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    print("\nNotice: MuScriptor models on HuggingFace require a HuggingFace User Token (HF_TOKEN).")
    print("If saved in Colab Secrets as 'HF_TOKEN', it is loaded automatically. Otherwise, enter below:")
    try:
        hf_token = getpass.getpass("HuggingFace Token (HF_TOKEN): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace Access Token configured successfully!")

print("\nInstalling system dependencies (FFmpeg, FluidSynth, and Tor IP-Masking Daemon)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth tor
!service tor start || /etc/init.d/tor start || true

print("\nCloning / Updating NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None

        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")

        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    !git checkout Stable 2>/dev/null || true
    !git pull origin Stable || !git pull || true
    print("\nInstalling Python dependencies (Future-proof live yt-dlp master branch)...")
    !pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" bgutil-ytdlp-pot-provider music21 soundfile mido
    if has_tpu:
        try:
            import torch_xla
        except ImportError:
            print("Installing torch_xla TPU support...")
            !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html
    !pip install -q --force-reinstall --no-deps muscriptor || !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


Checking hardware accelerator environment...
✓ NVIDIA GPU Detected: Tesla T4
Wed Aug 26 05:23:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                    

In [3]:
# @title 3. Setup MS Basic.sf3 SoundFont
import os
import shutil
import urllib.request

local_sf3_path = "/content/NC-Musical/MS Basic.sf3"
drive_sf3_path = "/content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3"
has_drive = os.path.exists("/content/drive/MyDrive")

target_sf3 = drive_sf3_path if has_drive else local_sf3_path

if not os.path.exists(target_sf3):
    print("Downloading MS Basic.sf3 SoundFont (~50MB)...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            os.makedirs(os.path.dirname(target_sf3), exist_ok=True)
            urllib.request.urlretrieve(url, target_sf3)
            print(f"✓ MS Basic.sf3 downloaded successfully to {target_sf3}!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print(f"✓ MS Basic.sf3 SoundFont found at: {target_sf3}")

# Sync to local workspace path if Drive was used
if has_drive and os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    shutil.copy2(drive_sf3_path, local_sf3_path)


✓ MS Basic.sf3 SoundFont found at: /content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3


In [ ]:
# @title 4. AI Music Transcription (Batch & Single Processing)
# @markdown ### 📥 Audio Input & Model Configuration
input_source = "Upload Files (.mp3, .wav, .flac)" # @param ["Upload Files (.mp3, .wav, .flac)", "YouTube URL(s)"]
youtube_urls = "" # @param {type:"string"}
model_size = "Large-v1 (Highest Quality & Polyphony)" # @param ["Large-v1 (Highest Quality & Polyphony)", "Small (Fastest)"]

# @markdown ### 🎛️ Transcription Focus
# @markdown Choose instrument focus (Songster multi-stems, full polyphonic mix, clean vocals, or piano grand staff):
transcription_mode = "Songster Multi-Stems (Vocals, Guitars, Drums, Bass, Piano)" # @param ["Songster Multi-Stems (Vocals, Guitars, Drums, Bass, Piano)", "Full Mix (All Instruments)", "Vocals Only (Clean Solo Voice)", "Piano Only (Grand Staff)"]

# @markdown ### 🎵 Musical Timing & Structure (Auto-Detect or Set)
# @markdown Simple, accurate musical controls:
time_signature = "Auto-Detect" # @param ["Auto-Detect", "4/4 (Common Time)", "3/4 (Waltz)", "2/4 (March)", "6/8 (Compound Duple)", "12/8 (Compound Quadruple)"]
bpm = "Auto-Detect" # @param ["Auto-Detect", "Manual (Use custom_bpm)"]
custom_bpm = 120.0 # @param {type:"number"}
key_signature = "Auto-Detect" # @param ["Auto-Detect", "C Major / A Minor", "G Major / E Minor", "F Major / D Minor", "D Major / B Minor", "Bb Major / G Minor", "A Major / F# Minor", "Eb Major / C Minor", "E Major / C# Minor", "Ab Major / F Minor", "B Major / G# Minor", "Db Major / Bb Minor"]
use_triplets = False # @param {type:"boolean"}

# @markdown ### 📤 Export Options
export_musicxml = True # @param {type:"boolean"}
export_chord_names = False # @param {type:"boolean"} 
download_zip_for_batch = False # @param {type:"boolean"}

import os
import sys
import re
import gc
import time
import io
import math
import zipfile
import subprocess
from pathlib import Path
import torch
import mido
import yt_dlp
import google.colab.files
from IPython.display import Audio, display

# Ensure repo root is on sys.path to access midi_processor and musicxml_exporter
for p in ["/content/NC-Musical", ".", os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    from midi_processor import (
        postprocess_midi,
        parse_time_signature_string,
        detect_time_signature_from_midi,
        detect_key_signature_from_midi,
        separate_multichannel_tracks,
        clean_monophonic_vocal,
        clean_monophonic_voice
    )
    from musicxml_exporter import export_midi_to_musicxml
except ImportError:
    # Fallback path inclusion
    sys.path.insert(0, "/content/NC-Musical")
    from midi_processor import (
        postprocess_midi,
        parse_time_signature_string,
        detect_time_signature_from_midi,
        detect_key_signature_from_midi,
        separate_multichannel_tracks,
        clean_monophonic_vocal,
        clean_monophonic_voice
    )
    from musicxml_exporter import export_midi_to_musicxml

def sanitize_filename(name: str) -> str:
    cleaned = re.sub(r'[\\/*?:"<>|]', "", name)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned if cleaned else "Transcription"

# ── 1. Resolve Target Instruments ─────────────────────────────────────────────
inst_list = None
if transcription_mode.startswith("Vocals Only"):
    inst_list = ["voice", "lead_vocal", "vocal"]
elif transcription_mode.startswith("Piano Only"):
    inst_list = ["piano", "keyboard"]

# ── 2. Gather Items into Processing Queue ─────────────────────────────────────
queue = []
if input_source.startswith("YouTube"):
    raw_input = youtube_urls.strip()
    if not raw_input:
        print("⚠️ Warning: YouTube URL(s) input is empty! Please paste at least one YouTube link.")
    else:
        urls = [u.strip() for u in re.split(r'[\n,\s]+', raw_input) if u.strip() and ("youtube.com" in u or "youtu.be" in u)]
        for url in urls:
            queue.append(("youtube", url, "YouTube Audio"))
else:
    print("📁 Please select audio file(s) to upload (.mp3, .wav, .flac, .ogg, .m4a):")
    uploaded = google.colab.files.upload()
    for fname in uploaded.keys():
        queue.append(("file", fname, Path(fname).stem))

if not queue:
    print("Queue is empty. Nothing to transcribe.")
else:
    print(f"✓ Found {len(queue)} track(s) in queue.")

    # ── 3. Persistent Hardware & Model Management ──────────────────────────────
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"

    # Map user-friendly model selection to internal MuScriptor size ('large', 'medium', 'small')
    model_size_str = str(model_size).strip()
    if model_size_str.lower() in ("small", "medium", "large"):
        weights_key = model_size_str.lower()
    elif "small" in model_size_str.lower():
        weights_key = "small"
    elif "medium" in model_size_str.lower():
        weights_key = "medium"
    else:
        weights_key = "large"

    global _CACHED_MODEL, _CACHED_MODEL_SIZE, _CACHED_DEVICE
    if '_CACHED_MODEL' not in globals():
        _CACHED_MODEL = None
        _CACHED_MODEL_SIZE = None
        _CACHED_DEVICE = None

    from muscriptor.transcription_model import TranscriptionModel
    from muscriptor.utils.audio import _read_wav_file
    from muscriptor.events import ProgressEvent
    from muscriptor.utils.auralization import synthesize

    if _CACHED_MODEL is not None and _CACHED_MODEL_SIZE == weights_key and _CACHED_DEVICE == device:
        print(f"✓ Reusing loaded '{weights_key}' model on {dev_name} (0s reload time, 0MB extra VRAM).")
        model = _CACHED_MODEL
    else:
        if _CACHED_MODEL is not None:
            print(f"Releasing previous '{_CACHED_MODEL_SIZE}' model from VRAM...")
            del _CACHED_MODEL
            _CACHED_MODEL = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"\nLoading MuScriptor AI Model ('{weights_key}' on {dev_name})...")
        model = TranscriptionModel.load_model(weights_path=weights_key, device=device)
        _CACHED_MODEL = model
        _CACHED_MODEL_SIZE = weights_key
        _CACHED_DEVICE = device
        print("✓ Model loaded successfully!\n")

    # YouTube download helper with Tor Proxy IP-Masking & PO-Token Botguard Bypass
    def fetch_youtube_audio(url, temp_stem="/content/yt_temp"):
        try:
            subprocess.run(["service", "tor", "start"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception:
            pass

        cookie_paths = ["/content/cookies.txt", "/content/drive/MyDrive/cookies.txt", "/content/drive/MyDrive/NC-Musical-Models/cookies.txt"]
        found_cookie = next((cp for cp in cookie_paths if os.path.exists(cp)), None)
        base_opts = {
            'format': 'bestaudio/best',
            'outtmpl': f'{temp_stem}.%(ext)s',
            'quiet': True,
            'no_warnings': True,
            'noplaylist': True,
            'socket_timeout': 30
        }

        strategies = [
            ("Tor Proxy IP-Masking", {
                'proxy': 'socks5://127.0.0.1:9050',
                'extractor_args': {'youtube': {'player_client': ['android', 'web']}}
            }),
            ("Tor Proxy + iOS Client", {
                'proxy': 'socks5://127.0.0.1:9050',
                'extractor_args': {'youtube': {'player_client': ['ios', 'mweb']}}
            }),
            ("PO-Token Provider + Web Safari", {
                'extractor_args': {'youtube': {'player_client': ['web_safari', 'web']}}
            }),
            ("Web Safari Client Direct", {
                'extractor_args': {'youtube': {'player_client': ['web_safari']}}
            }),
            ("Standard yt-dlp Direct", {})
        ]

        if found_cookie:
            strategies.insert(0, ("Tor Proxy + Session Cookies", {
                'proxy': 'socks5://127.0.0.1:9050',
                'cookiefile': found_cookie
            }))
            strategies.insert(1, ("Direct yt-dlp + Session Cookies", {
                'cookiefile': found_cookie
            }))

        last_err = None
        title = "Unknown Song"
        for name, extra_opts in strategies:
            opts = {**base_opts, **extra_opts}
            try:
                for f in Path("/content").glob("yt_temp.*"):
                    try: os.remove(f)
                    except: pass
                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(url, download=True)
                    if info and 'title' in info: title = info['title']
                dl_files = list(Path("/content").glob("yt_temp.*"))
                if dl_files:
                    return str(dl_files[0]), title, None
            except Exception as e:
                last_err = e
                if 'proxy' in extra_opts:
                    try:
                        subprocess.run(["pkill", "-HUP", "tor"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    except Exception:
                        pass
        return None, title, last_err

    def _fmt_time(s):
        return f"{int(s//60)}m {int(s%60):02d}s" if s >= 60 else f"{s:.0f}s"

    def _draw_progress(completed, total, elapsed, note_count):
        pct = int(completed / total * 100) if total > 0 else 0
        bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
        eta = (elapsed / max(completed, 1)) * (total - completed) if completed > 0 else 0
        print(f"\r  [{bar}] {pct:3d}%  |  Elapsed: {_fmt_time(elapsed)}  |  ETA: {_fmt_time(eta)}  |  Notes: {note_count}    ", end="", flush=True)

    # ── 4. Execute Batch Queue with Fail-Safe AI & Musical Post-Processing ─────
    batch_results = []
    output_dir = Path("/content/NC_Musical_Outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    for idx, (itype, isrc, ititle) in enumerate(queue, 1):
        print("═" * 70)
        print(f"🎵 [Track {idx}/{len(queue)}] Processing: {isrc if itype == 'youtube' else isrc}")
        print("═" * 70)

        wav_input = "/content/current_input.wav"
        if os.path.exists(wav_input):
            os.remove(wav_input)

        song_title = ititle
        if itype == "youtube":
            print("Downloading YouTube audio...")
            raw_audio, fetched_title, err = fetch_youtube_audio(isrc)
            if not raw_audio:
                print(f"❌ Failed to download audio for {isrc}: {err}")
                continue
            song_title = sanitize_filename(fetched_title)
            print(f"✓ Title: {song_title}")
            print("Converting to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(raw_audio), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])
            try: os.remove(raw_audio)
            except: pass
        else:
            song_title = sanitize_filename(ititle)
            print(f"✓ Title: {song_title}")
            print(f"Converting {isrc} to 16kHz mono WAV...")
            subprocess.run(["ffmpeg", "-y", "-i", str(isrc), "-ac", "1", "-ar", "16000", str(wav_input), "-loglevel", "quiet"])

        if not os.path.exists(wav_input):
            print(f"❌ Failed to prepare audio for {song_title}.")
            continue

        with open(wav_input, "rb") as f:
            wav, sr = _read_wav_file(f)

        print("Transcribing audio notes...")
        events = []
        start_t = time.time()
        devnull = open(os.devnull, 'w')
        real_stdout = sys.stdout
        try:
            with torch.inference_mode():
                for ev in model.transcribe((wav, sr), instruments=inst_list, batch_size=1):
                    sys.stdout = real_stdout
                    if isinstance(ev, ProgressEvent):
                        if ev.total > 0:
                            _draw_progress(ev.completed, ev.total, time.time() - start_t, len(events))
                    else:
                        events.append(ev)
                    sys.stdout = devnull
        finally:
            sys.stdout = real_stdout
            devnull.close()

        elapsed = time.time() - start_t
        print(f"\n\n✅ Complete in {_fmt_time(elapsed)} — {len(events)} note events extracted.")

        # Resolve BPM (Auto-Detect or Manual)
        final_bpm = float(custom_bpm) if ("Manual" in bpm) else 120.0
        if "Auto-Detect" in bpm:
            try:
                print("Attempting AI beat tracking & tempo detection...")
                if hasattr(model, 'transcribe_and_postprocess'):
                    _, ai_grid = model.transcribe_and_postprocess(wav_input, detect_tempo="best-effort", quantize=False)
                    if ai_grid is not None and getattr(ai_grid, 'bpm', None):
                        final_bpm = float(ai_grid.bpm)
                        print(f"✓ AI Detected Tempo: {final_bpm:.1f} BPM")
            except Exception as tempo_err:
                print(f"⚠️ AI beat-tracking notice: {tempo_err} -> Using adaptive tempo {final_bpm:.1f} BPM")

        # Convert raw notes to MIDI bytes
        raw_midi_bytes = model.events_to_midi_bytes(iter(events))
        raw_mid = mido.MidiFile(file=io.BytesIO(raw_midi_bytes))

        # Resolve Time Signature (Auto-Detect or User Choice)
        if "Auto-Detect" in time_signature:
            eff_ts_num, eff_ts_denom = detect_time_signature_from_midi(raw_mid)
        else:
            eff_ts_num, eff_ts_denom = parse_time_signature_string(time_signature)

        # Resolve Key Signature (Auto-Detect or User Choice)
        if "Auto-Detect" in key_signature:
            resolved_key = detect_key_signature_from_midi(raw_mid)
        else:
            resolved_key = key_signature.split()[0]

        # Resolve Quantization & Triplets Grid
        # use_triplets=False -> grid_div=4 (straight 16th notes: no tuplets, clean MuseScore single-voice)
        # use_triplets=True  -> grid_div=6 (allows triplet sixteenths)
        grid_div = 6 if use_triplets else 4

        # Apply robust musical post-processing
        print("Applying post-processing (Grand Staff Piano, Clean Voice 1, Full Kit Drums, Guitar Staves)...")
        processed_mid, final_midi_bytes, meta = postprocess_midi(
            raw_mid,
            bpm=final_bpm,
            time_sig_num=eff_ts_num,
            time_sig_denom=eff_ts_denom,
            key_signature=resolved_key,
            quantize=True,
            grid_div=grid_div,
            quantize_strength=1.0,
            fill_gaps_beats=0.25,
            trim_leading_bars=False,
            single_voice=True,
            separate_channels=True
        )

        resolved_key = meta.get("key_signature", resolved_key)

        midi_path = output_dir / f"{song_title}.mid"
        with open(midi_path, "wb") as f:
            f.write(final_midi_bytes)

        print(f"💾 MIDI Saved: {midi_path.name}")
        print(f" • Time Signature: {eff_ts_num}/{eff_ts_denom} | Key: {resolved_key} | Tempo: {final_bpm:.1f} BPM")
        print(f" • Quantization:   {'Triplet 1/24 grid' if use_triplets else 'Straight 1/16 grid (No Tuplets)'}")
        print(f" • MuseScore Voice: Single Voice 1 for Vocals (No 2-voicing); Grand Staff for Piano; Full Drums")

        # Export MusicXML for native MuseScore rendering
        xml_path = None
        if export_musicxml:
            xml_path = output_dir / f"{song_title}.musicxml"
            try:
                export_midi_to_musicxml(
                    processed_mid,
                    output_xml_path=xml_path,
                    title=song_title,
                    bpm=final_bpm,
                    time_sig_num=eff_ts_num,
                    time_sig_denom=eff_ts_denom,
                    key_signature=resolved_key
                )
                print(f"🎼 MusicXML Saved: {xml_path.name} (Ready for MuseScore Grand Staff & Clean Voices)")
            except Exception as xml_err:
                print(f"⚠️ MusicXML export notice: {xml_err}")

        # Chord Analysis (optional)
        chord_path = None
        if export_chord_names:
            chord_path = output_dir / f"{song_title}_chords.txt"
            try:
                from collections import defaultdict
                from music21 import converter, chord as m21chord
                score = converter.parse(str(midi_path))

                time_groups = defaultdict(list)
                for el in score.flatten().notes:
                    start = float(el.offset)
                    end = start + float(el.quarterLength)
                    pitches = [el.pitch] if el.isNote else el.pitches
                    for p in pitches:
                        time_groups[start].append((1, p))
                        time_groups[end].append((-1, p))

                lines = [
                    f"CHORD ANALYSIS — {song_title}",
                    f"Key Signature: {resolved_key} | Time Signature: {eff_ts_num}/{eff_ts_denom} | Tempo: {final_bpm:.1f} BPM",
                    "=" * 60
                ]
                active_pitches = defaultdict(int)
                prev_label = None
                prev_pitch_set = None

                for t in sorted(time_groups.keys()):
                    for event_type, p in sorted(time_groups[t], key=lambda x: x[0]):
                        if event_type == -1:
                            active_pitches[p] -= 1
                            if active_pitches[p] <= 0:
                                del active_pitches[p]
                        elif event_type == 1:
                            active_pitches[p] += 1

                    if active_pitches:
                        current_pitch_set = frozenset(active_pitches.keys())
                        if current_pitch_set != prev_pitch_set:
                            c = m21chord.Chord(sorted(list(current_pitch_set)))
                            ts = f"{int(t//60)}:{t%60:05.2f}"
                            try:
                                label = f"{c.root().name} {c.commonName}" if c.root() else " ".join(p.name for p in c.pitches)
                            except Exception:
                                label = " ".join(p.name for p in c.pitches)
                            if label != prev_label:
                                lines.append(f"[{ts}]  {label}")
                                prev_label = label
                            prev_pitch_set = current_pitch_set

                with open(chord_path, "w", encoding="utf-8") as f:
                    f.write("\n".join(lines))
                print(f"🎵 Chords Saved: {chord_path.name}")
            except Exception as e:
                print(f"Chord analysis notice: {e}")

        # Audio preview
        preview_wav = "/content/preview.wav"
        sf3_path = "/content/NC-Musical/MS Basic.sf3"
        try:
            synthesize(str(midi_path), preview_wav, soundfont_path=sf3_path if os.path.exists(sf3_path) else None)
            print(f"🎧 Audio Preview ({song_title}):")
            display(Audio(preview_wav))
        except Exception:
            pass

        # Auto download individual files (if not packaging as ZIP)
        if not download_zip_for_batch:
            try:
                google.colab.files.download(str(midi_path))
                if xml_path and xml_path.exists():
                    time.sleep(0.5)
                    google.colab.files.download(str(xml_path))
                if chord_path and chord_path.exists():
                    time.sleep(0.5)
                    google.colab.files.download(str(chord_path))
            except Exception as e:
                print(f"Download notice: {e}")

        batch_results.append({
            "title": song_title,
            "midi": str(midi_path),
            "musicxml": str(xml_path) if xml_path and xml_path.exists() else None,
            "chords": str(chord_path) if chord_path and chord_path.exists() else None,
            "notes": len(events),
            "key": resolved_key,
            "meter": f"{eff_ts_num}/{eff_ts_denom}",
            "bpm": f"{final_bpm:.1f}",
            "time": _fmt_time(elapsed)
        })

        del wav, events
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── 5. Batch ZIP Package ──────────────────────────────────────────────────
    if download_zip_for_batch and batch_results:
        zip_path = "/content/NC_Musical_Results.zip"
        print("\n" + "═" * 70)
        print("📦 Packaging all transcriptions into ZIP archive...")
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for item in batch_results:
                if os.path.exists(item['midi']):
                    zipf.write(item['midi'], arcname=Path(item['midi']).name)
                if item.get('musicxml') and os.path.exists(item['musicxml']):
                    zipf.write(item['musicxml'], arcname=Path(item['musicxml']).name)
                if item.get('chords') and os.path.exists(item['chords']):
                    zipf.write(item['chords'], arcname=Path(item['chords']).name)
        print(f"✓ ZIP archive created: {zip_path}")
        google.colab.files.download(zip_path)

    # ── 6. Summary Table ──────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("📊 BATCH TRANSCRIPTION SUMMARY")
    print("═" * 70)
    for r in batch_results:
        print(f" • {r['title']} | Notes: {r['notes']:,} | Key: {r['key']} | Meter: {r['meter']} | Tempo: {r['bpm']} BPM | Time: {r['time']}")
    print(f"\nAll results saved to directory: {output_dir}\n")
